# Session 31 SQL DML Commands

## Introduction to SQL DML Commands

In the language of SQL, a database is commonly referred to as a **Schema**. A schema acts as a logical container that stores database objects such as tables, views, functions, indexes, and more.

In this session, the primary focus will be on **DML (Data Manipulation Language)** commands in SQL. DML commands are used to interact with the data stored inside tables. These commands allow us to insert, update, delete, and retrieve records from a database.

In the earlier sessions, all database operations were performed using **pgAdmin4**. In this session, however, the work is being carried out using DataGrip for a change. At various points, comparisons between pgAdmin4 and DataGrip may also be discussed to understand the advantages and usability differences between the two tools.

Before beginning the session, all previously created tables were dropped, and a fresh table named `users` was created.

---

## Creating the `users` Table

```sql
CREATE TABLE users (
    user_id SERIAL PRIMARY KEY,
    name VARCHAR (255),
    email VARCHAR (255) NOT NULL UNIQUE,
    password VARCHAR (255) NOT NULL
);
```

### Understanding the Table Structure

The `users` table contains the following columns:

| Column Name | Data Type      | Constraint       |
| ----------- | -------------- | ---------------- |
| `user_id`   | `SERIAL`       | Primary Key      |
| `name`      | `VARCHAR(255)` | Optional         |
| `email`     | `VARCHAR(255)` | NOT NULL, UNIQUE |
| `password`  | `VARCHAR(255)` | NOT NULL         |

### Important Observations

* The `SERIAL` pseudo-type in PostgreSQL automatically generates incrementing integer values.
* `user_id` acts as the primary key of the table.
* The `email` column cannot contain duplicate values because of the `UNIQUE` constraint.
* Both `email` and `password` cannot contain `NULL` values due to the `NOT NULL` constraint.

---

## Creation commands

## INSERT Command in PostgreSQL

The `INSERT` statement is used to add new rows into a table.

The following queries are all valid in PostgreSQL:

```sql
INSERT INTO users (user_id, name, email, password)
VALUES (DEFAULT, 'neji', 'neji@email.com', '1234');

INSERT INTO users (name, email, password)
VALUES ('moneyk', 'moneyl@email.com', '1234');

INSERT INTO public.users (name, email, password)
VALUES ('ankit', 'ankit@email.com', '1234');

INSERT INTO learningdb.public.users (name, email, password)
VALUES ('hola', 'hola@email.com', '1234');
```

---

## Understanding Object Referencing in PostgreSQL

Notice that in each query, the same `users` table has been referenced in different ways:

| Reference Style           | Meaning                                  |
| ------------------------- | ---------------------------------------- |
| `users`                   | Table name only                          |
| `public.users`            | Schema name + table name                 |
| `learningdb.public.users` | Database name + schema name + table name |

All of these queries work correctly because PostgreSQL internally organizes database objects in a hierarchical structure:

```text
Database → Schema → Table
```

This structure can also be observed in the Database Explorer or Object Explorer of tools like DataGrip and pgAdmin4.

---

## Understanding the `SERIAL` Pseudo-Type

In most of the previous queries, no value was explicitly provided for `user_id`.

For example:

```sql
INSERT INTO users (name, email, password)
VALUES ('moneyk', 'moneyl@email.com', '1234');
```

This works because PostgreSQL automatically generates values for the `user_id` column using the `SERIAL` pseudo-type.

### How `SERIAL` Works

The `SERIAL` pseudo-type internally:

1. Creates an integer column
2. Creates a sequence object
3. Automatically increments values during insertion

As a result, each newly inserted row receives the next available integer value automatically.

---

## Manually Inserting Values into a SERIAL Column

It is also possible to manually provide values for a `SERIAL` column.

```sql
INSERT INTO learningdb.public.users
VALUES (8, 'gola', 'gola@email.com', '1234');
```

In this query:

* `8` is manually inserted into the `user_id` column.
* PostgreSQL accepts this query as long as the inserted value does not violate constraints such as uniqueness.

---

## NOT NULL Constraint Violation

Unlike MySQL, PostgreSQL strictly enforces `NOT NULL` constraints.

The following query is **not valid**:

```sql
INSERT INTO learningdb.public.users (name, email)
VALUES ('trola', 'trola@email.com');
```

### Error Message

```text
[2026-05-28 17:02:26] [23502] ERROR: null value in column "password" of relation "users" violates not-null constraint
[2026-05-28 17:02:26] Detail: Failing row contains (4, trola, trola@email.com, null).
```

### Reason for the Error

The `password` column was defined using the `NOT NULL` constraint:

```sql
password VARCHAR (255) NOT NULL
```

Since no value was provided for the `password` column, PostgreSQL attempted to insert `NULL`, which violated the constraint.

---

## Column Order Flexibility During INSERT

When column names are explicitly specified in the `INSERT` statement, the order of values depends entirely on the order of the mentioned columns.

For example:

```sql
INSERT INTO users (password, name, email)
VALUES ('1234', 'kakashi', 'kakashi@email.com');
```

This query works correctly because:

* PostgreSQL maps values according to the specified column order.
* The physical order of columns inside the table does not matter.

This provides flexibility while inserting data into tables.

---

## Inserting Multiple Rows at Once

PostgreSQL also allows inserting multiple rows using a single `INSERT` statement.

```sql
INSERT INTO users
VALUES (DEFAULT, 'sasuke', 'sasuke@email.com', '1234'),
       (DEFAULT, 'sakura', 'sakura@email.com', '1234'),
       (DEFAULT, 'narut', 'narut@email.com', '1234');
```

### Advantages of Multi-Row INSERT

* Reduces the number of database calls
* Improves performance
* Makes bulk insertion easier and cleaner
* Frequently used during data migration and seeding operations

In this query:

* Three records are inserted simultaneously.
* `DEFAULT` is used for the `user_id` column so that PostgreSQL automatically generates the IDs.

## Retrieval Commands

Retrieval commands in SQL are primarily used to fetch data from one or more tables. The most commonly used retrieval command is the `SELECT` statement.

---

## SELECT All

A basic `SELECT` query to display all rows from a table is:

```sql id="s4f8d2"
SELECT * FROM smartphones;
```

### Understanding the `*` Symbol

A common misconception is that the `*` symbol refers to selecting all rows from a table. In reality:

* `*` represents **all columns** of the table.
* The rows are returned because no filtering condition has been applied.

So, the query:

```sql id="a0l2d9"
SELECT * FROM smartphones;
```

actually means:

```text id="8m1k5v"
Select all columns from the smartphones table.
```

---

## Why `SELECT * FROM smartphones WHERE 1` Does Not Work in PostgreSQL

In MySQL, the following query works:

```sql id="m3n7c1"
SELECT * FROM smartphones WHERE 1;
```

### What Does This Query Mean?

In MySQL:

* `1` is treated as a boolean expression.
* Since `1` evaluates to `TRUE`, the query effectively becomes:

```text id="t4v9x6"
Return all rows where the condition is TRUE.
```

As a result, all rows are returned.

---

### Why It Fails in PostgreSQL

PostgreSQL has stricter type checking compared to MySQL.

In PostgreSQL:

* The `WHERE` clause must contain a proper boolean expression.
* Integer values like `1` are not automatically converted into boolean values.

Therefore, PostgreSQL throws an error because:

```text id="p6r2z0"
1 is an integer, not a boolean condition.
```

The PostgreSQL-compatible version would be:

```sql id="b9u4q8"
SELECT * FROM smartphones WHERE TRUE;
```

---

## SELECT Filter

Instead of retrieving all columns, specific columns can also be selected.

```sql id="q7f1y5"
SELECT model, price, rating FROM smartphones;
```

### Important Observation

* The number of rows returned will remain the same.
* Only the selected columns (`model`, `price`, and `rating`) will appear in the output table.

This process is known as **column filtering** or **projection**.

### Column Sequence Flexibility

The order of columns inside the query does not need to match the physical order of columns inside the table.

For example:

```sql id="v2g8k1"
SELECT rating, model, price FROM smartphones;
```

is completely valid.

The output columns will appear in the same order as written in the query.

---

## Aliasing of Column Names

Aliases are temporary names given to columns or tables in the output.

```sql id="r8c3m6"
SELECT os AS "Operating System", model, battery_capacity
FROM smartphones;
```

### Why Use Aliases?

Aliases improve readability of the output table.

For example:

* `os` becomes `Operating System`
* This makes the result easier to understand for end users.

---

### Why Double Quotes Are Used

In PostgreSQL:

* Double quotes (`" "`) are used for identifiers such as:

  * Column aliases
  * Table names
  * Schema names

Since the alias contains a space (`Operating System`), double quotes are required.

---

### Why Single Quotes Will Not Work

The following query is incorrect:

```sql id="h1s7e3"
SELECT os AS 'Operating System'
FROM smartphones;
```

This is because:

* Single quotes (`' '`) represent **string literals** in SQL.
* PostgreSQL interprets `'Operating System'` as plain text data, not as a column alias.

Therefore:

```text id="u5n0l4"
Double quotes → identifiers
Single quotes → string values/constants
```

This distinction is extremely important in PostgreSQL.

---

## Mathematical Expressions on Columns

SQL also allows mathematical operations and functions directly on columns.

```sql id="e3x7p9"
SELECT model,
sqrt(resolution_width*resolution_width + resolution_height*resolution_height)/screen_size AS ppi
FROM smartphones;
```

### Understanding the Query

In this query:

* A mathematical expression is applied on existing columns.
* A new calculated column named `ppi` is generated in the output.

### What is PPI?

PPI stands for:

```text id="k7t2v5"
Pixels Per Inch
```

It is a measurement of screen pixel density.

The formula calculates the diagonal pixel resolution using the Pythagorean theorem and divides it by the screen size.

The newly generated `ppi` column exists only in the query result and is not permanently stored in the table.

---

## Constants in Retrieval Queries

Sometimes constant values are added to query outputs for labeling or categorization purposes.

```sql id="w8m4c2"
SELECT model, 'smartphone' AS "type"
FROM smartphones;
```

### Understanding the Query

In this query:

* `model` comes from the table.
* `'smartphone'` is a constant string value added for every row.
* `"type"` is the alias name of the generated column.

The output table will contain:

| model      | type       |
| ---------- | ---------- |
| Galaxy S24 | smartphone |
| iPhone 15  | smartphone |

---

## Single Quotes vs Double Quotes in SQL

This is one of the most important syntax distinctions in PostgreSQL.

| Symbol | Purpose                                     |
| ------ | ------------------------------------------- |
| `' '`  | String literals / constant values           |
| `" "`  | Identifiers such as aliases or column names |

### Examples

#### String Literal

```sql id="j9q5r2"
SELECT 'hello';
```

Here:

* `'hello'` is treated as text data.

---

#### Identifier / Alias

```sql id="l6d8u1"
SELECT model AS "Phone Model"
FROM smartphones;
```

Here:

* `"Phone Model"` is treated as a column alias.

---

## DISTINCT

The `DISTINCT` keyword is used to remove duplicate values from the result set.

```sql id="c4n9t7"
SELECT DISTINCT (smartphones.brand_name) AS brand_names
FROM smartphones;
```

### What Does This Query Do?

This query returns:

* Only unique brand names
* Duplicate entries are removed automatically

For example:

```text id="g8y2m5"
samsung
apple
xiaomi
oneplus
```

instead of showing repeated values multiple times.

---

## Unique Combinations

`DISTINCT` can also be applied on combinations of multiple columns.

```sql id="n5w1z8"
SELECT DISTINCT smartphones.brand_name, smartphones.processor_brand
FROM smartphones;
```

### Understanding the Query

This query returns only unique combinations of:

* `brand_name`
* `processor_brand`

For example:

| brand_name | processor_brand |
| ---------- | --------------- |
| samsung    | exynos          |
| samsung    | snapdragon      |
| apple      | bionic          |

If the same combination appears multiple times in the table, it will be shown only once in the result.

---

## Filter Rows Using WHERE Clause

The `WHERE` clause is used to filter rows based on conditions.

---

### Find All Samsung Phones

```sql id="f2u7r3"
SELECT * FROM smartphones
WHERE brand_name = 'samsung';
```

This query returns only those rows where the brand name is `samsung`.

---

### Find All Phones With Price Greater Than 50000

```sql id="z1k6v9"
SELECT * FROM smartphones
WHERE price > 50000;
```

This query filters all smartphones whose price is greater than `50000`.

---

## BETWEEN Operator

The `BETWEEN` operator is used to filter values within a range.

---

### Find All Phones With 10000 < Price < 20000

A primitive approach would be:

```sql id="y7n4m1"
SELECT * FROM smartphones
WHERE price > 10000 AND price < 20000;
```

A cleaner approach uses `BETWEEN`:

```sql id="d3f8p5"
SELECT * FROM smartphones
WHERE price BETWEEN 10000 AND 20000;
```

### Important Note About BETWEEN

`BETWEEN` is inclusive in SQL.

Meaning:

```text id="r4w9k2"
10000 and 20000 are both included in the range.
```

So internally, it behaves like:

```sql id="o8u1h6"
price >= 10000 AND price <= 20000
```

---

### Find Phones With Rating > 80, Price < 25000, and Snapdragon Processor

```sql id="t5e2v7"
SELECT * FROM smartphones
WHERE price < 25000
AND rating > 80
AND processor_brand = 'snapdragon';
```

This query applies multiple filtering conditions simultaneously using the `AND` operator.

---

### Find All Samsung Phones With RAM Greater Than or Equal to 8 GB

```sql id="u6m3x8"
SELECT * FROM smartphones
WHERE brand_name = 'samsung'
AND ram_capacity >= 8.0;
```

---

### Find All Samsung Phones With Snapdragon Processor

```sql id="i4p7q1"
SELECT * FROM smartphones
WHERE brand_name = 'samsung'
AND processor_brand = 'snapdragon';
```

---

## Order of Query Execution

A `SELECT` query may contain many clauses. Internally, SQL executes them in a specific order.

The execution order is:

```text id="b7s2n9"
FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → DISTINCT → ORDER BY
```

A quick mnemonic to remember this order is:

```text id="v1c5e8"
Frand John's Wicked Grave Haunts Several Dull Owls
```

### Why Is This Important?

Understanding execution order helps in:

* Debugging queries
* Understanding filtering behavior
* Writing optimized SQL queries
* Understanding aggregation logic

---

### Find Brands That Sold At Least One Phone Above 100000

```sql id="x9l4d2"
SELECT DISTINCT (brand_name)
FROM smartphones
WHERE price > 100000;
```

### Execution Understanding

1. Data is taken from `smartphones`
2. Rows with `price > 100000` are filtered
3. `brand_name` is selected
4. Duplicate brands are removed using `DISTINCT`

---

## IN and NOT IN Operators

---

### Primitive Approach Using OR

```sql id="p8f3k6"
SELECT *
FROM smartphones
WHERE processor_brand = 'snapdragon'
   OR processor_brand = 'exynox'
   OR processor_brand = 'bionic';
```

### Problem With This Approach

If more processor brands need to be added later:

* More `OR` conditions must be written repeatedly
* The query becomes lengthy and less readable

---

## Using the IN Operator

```sql id="m2q7u5"
SELECT *
FROM smartphones
WHERE processor_brand IN ('snapdragon', 'exynos', 'bionic');
```

### Advantages of IN

* Cleaner syntax
* Better readability
* Easier maintenance
* Convenient for multiple matching conditions

If another processor brand needs to be added later, it can simply be included inside the list.

---

## Using NOT IN

To exclude certain values, the `NOT IN` operator can be used.

```sql id="s7v1e4"
SELECT *
FROM smartphones
WHERE processor_brand NOT IN ('snapdragon', 'exynos', 'bionic');
```

### What Does This Query Do?

This query returns all smartphones whose processor brand is:

* Not `snapdragon`
* Not `exynos`
* Not `bionic`

In other words, it excludes all rows matching the given list of values.

## UPDATE

The `UPDATE` statement is used to modify existing data stored inside a table. It allows changes to be made:

* To specific rows
* To specific columns
* Based on certain conditions

Unlike retrieval queries, `UPDATE` permanently changes the actual data stored in the database.

---

## Updating Existing Values

Suppose there are some rows where the `processor_brand` has been written as `'mediatek'`, and we want to replace it with `'dimensity'`.

The reason for this correction is that:

* **MediaTek** is the company name
* **Dimensity** is the processor series or chipset branding

Before performing the update, it is always a good practice to first verify the affected rows.

```sql id="m8f2x7"
SELECT *
FROM smartphones
WHERE processor_brand = 'mediatek';
```

After running this query, suppose one matching row is found.

The update can then be performed as follows:

```sql id="v4k9p1"
UPDATE smartphones
SET processor_brand = 'dimensity'
WHERE processor_brand = 'mediatek';
```

---

## Understanding the UPDATE Query

### Components of the Query

| Clause                               | Purpose                                |
| ------------------------------------ | -------------------------------------- |
| `UPDATE smartphones`                 | Specifies the table to modify          |
| `SET processor_brand = 'dimensity'`  | Defines the new value                  |
| `WHERE processor_brand = 'mediatek'` | Specifies which rows should be updated |

---

## Important Observation About UPDATE

The changes made using the `UPDATE` statement are:

* Permanent
* Applied directly to the database
* Not limited only to the output table

Therefore, extreme caution should be exercised while using `UPDATE`, especially when writing the `WHERE` clause.

---

## Danger of Missing WHERE Clause

Consider the following query:

```sql id="u3n6q8"
UPDATE smartphones
SET processor_brand = 'dimensity';
```

Since no `WHERE` condition is provided:

```text id="c7v1m4"
Every row of the table will be updated.
```

This means the `processor_brand` value of all smartphones would become `'dimensity'`.

Hence, the `WHERE` clause is extremely important in selective updates.

---

## DELETE

The `DELETE` statement is used to remove rows from a table.

Unlike `DROP TABLE`, which removes the entire table structure, `DELETE` removes only selected records while keeping the table intact.

---

## Deleting Rows Based on a Condition

Suppose we want to delete smartphones whose price is greater than `200000`, considering them as outliers.

Before deleting, it is good practice to first inspect the rows:

```sql id="e1p7z5"
SELECT *
FROM smartphones
WHERE price > 200000;
```

After verification, the rows can be deleted using:

```sql id="r9w4k2"
DELETE
FROM smartphones
WHERE price > 200000;
```

---

## Understanding the DELETE Query

### Components of the Query

| Clause                    | Purpose                        |
| ------------------------- | ------------------------------ |
| `DELETE FROM smartphones` | Specifies the table            |
| `WHERE price > 200000`    | Specifies which rows to remove |

After execution:

* Only rows satisfying the condition are deleted
* The table structure remains unchanged

---

## DELETE Using Multiple Conditions

Rows can also be deleted using multiple filtering conditions.

```sql id="q5x8d1"
DELETE
FROM smartphones
WHERE primary_camera_rear > 150
AND brand_name = 'samsung';
```

### What Does This Query Do?

This query deletes only those rows where:

* `primary_camera_rear > 150`
* AND `brand_name = 'samsung'`

Both conditions must be satisfied simultaneously.

---

## Danger of DELETE Without WHERE

Consider the following query:

```sql id="n2v7f6"
DELETE FROM smartphones;
```

Since no `WHERE` clause is present:

```text id="j8m3p5"
All rows from the table will be deleted.
```

However:

* The table itself will still exist
* Only the stored data will be removed

This is different from:

```sql id="y4u1k9"
DROP TABLE smartphones;
```

which removes the entire table structure permanently.

---

## Difference Between ALTER and UPDATE

A very common confusion in SQL is between `ALTER` and `UPDATE`.

Although both modify something, they operate at completely different levels.

---

## ALTER vs UPDATE

| Feature     | ALTER                           | UPDATE                 |
| ----------- | ------------------------------- | ---------------------- |
| Purpose     | Modifies table structure        | Modifies table data    |
| Operates On | Schema/Table definition         | Rows and column values |
| Affects     | Columns, constraints, datatypes | Stored records         |
| Category    | DDL Command                     | DML Command            |

---

## ALTER Command Example

```sql id="d6r2u8"
ALTER TABLE smartphones
ADD COLUMN launch_year INT;
```

### What Does This Do?

This changes the structure of the table by adding a new column.

---

## UPDATE Command Example

```sql id="f3w9k1"
UPDATE smartphones
SET launch_year = 2025
WHERE brand_name = 'samsung';
```

### What Does This Do?

This changes the values stored inside the rows.

The structure of the table remains unchanged.

---

## CRUD Operations in PostgreSQL

Up until this point, the fundamental CRUD operations using PostgreSQL have been covered.

CRUD stands for:

| Operation | SQL Command |
| --------- | ----------- |
| Create    | `INSERT`    |
| Retrieve  | `SELECT`    |
| Update    | `UPDATE`    |
| Delete    | `DELETE`    |

These operations form the foundation of database interaction in SQL.

Although the core CRUD operations are now covered, more advanced retrieval queries and concepts related to the `SELECT` statement will still be explored later on.